In [ ]:
import sysfrom pathlib import Path# Handle multiple working directory scenarios# Case 1: Running from project root# Case 2: Running from notebooks/ directory# Case 3: Running as a Jupyter notebookcurrent_file = Path(__file__).resolve() if '__file__' in dir() else Path.cwd()if current_file.is_file():    # We're running a notebook file    notebook_dir = current_file.parent    project_root = notebook_dir.parentelse:    # We're in a directory    project_root = Path.cwd()    # If we're in notebooks/ subdirectory, go up one level    if project_root.name == 'notebooks':        project_root = project_root.parent# Add project root to path if not already thereif str(project_root) not in sys.path:    sys.path.insert(0, str(project_root))# Verify src module existssrc_path = project_root / 'src'if not src_path.exists():    raise RuntimeError(f"Could not find src/ directory. Project root: {project_root}")print(f"✓ Added {project_root} to sys.path")

# Transformers vs Diffusion: Shared Foundation

This notebook demonstrates the core thesis: **both autoregressive (GPT) and diffusion (masked LM) generation paradigms rest on the same multi-head self-attention block.**

We'll:
1. Verify both models use identical `MultiHeadSelfAttention`
2. Train both on the same toy corpus
3. Compare generation dynamics (left-to-right vs iterative unmasking)
4. Show how the `causal` flag is the only architectural difference

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from src.transformer.gpt import TinyGPT
from src.diffusion.masked_lm import TinyMaskedLM
from src.common.tokenizer import CharTokenizer
from src.common.attention import MultiHeadSelfAttention

# Set random seed for reproducibility
torch.manual_seed(42)

print("✓ Imports successful")

## Part 1: Verify Shared Attention Module

First, let's confirm that both models use the exact same `MultiHeadSelfAttention` class, with only the `causal` flag differing.

In [ ]:
# Create both models
gpt = TinyGPT(vocab_size=100, d_model=64, n_layers=2, n_heads=4)
masked_lm = TinyMaskedLM(vocab_size=100, d_model=64, n_layers=2, n_heads=4)

print("=" * 60)
print("SHARED ATTENTION MODULE VERIFICATION")
print("=" * 60)

# Check the attention modules
gpt_attn = gpt.blocks[0].attn
mlm_attn = masked_lm.blocks[0].attn

print(f"\nGPT attention type: {type(gpt_attn).__name__}")
print(f"Masked LM attention type: {type(mlm_attn).__name__}")
print(f"\nSame class? {type(gpt_attn) is type(mlm_attn)}")

print(f"\nGPT attention causal flag: {gpt_attn.causal}")
print(f"Masked LM attention causal flag: {mlm_attn.causal}")

# Check embeddings
print(f"\nGPT embeddings type: {type(gpt.embed).__name__}")
print(f"Masked LM embeddings type: {type(masked_lm.embed).__name__}")
print(f"Same embeddings class? {type(gpt.embed) is type(masked_lm.embed)}")

print(f"\n✓ THESIS VERIFIED: Both models share MultiHeadSelfAttention")
print(f"  - GPT: causal=True (left-to-right)")
print(f"  - Masked LM: causal=False (bidirectional)")
print(f"  - Everything else identical")

## Part 2: Prepare Training Data

Create a toy corpus and set up tokenization.

In [ ]:
# Toy corpus
corpus = "hello hello hello world world"
print(f"Training corpus: '{corpus}'")

# Tokenizer
tokenizer = CharTokenizer(corpus)
print(f"\nVocabulary size: {tokenizer.vocab_size}")
print(f"Unique characters: {tokenizer.chars}")

# Tokenize
token_ids = torch.tensor([tokenizer.encode(corpus)])
print(f"\nTokenized: {token_ids[0].tolist()}")
print(f"Shape: {token_ids.shape}")

# For masked LM, shift tokens by 1 (since 0 = MASK)
token_ids_shifted = token_ids + 1
print(f"\nShifted (for masked LM): {token_ids_shifted[0].tolist()}")

## Part 3: Train Both Models

Train both models on the same corpus using their respective objectives.

In [ ]:
# Create fresh models for training
gpt = TinyGPT(vocab_size=tokenizer.vocab_size, d_model=64, n_layers=2, n_heads=4)
masked_lm = TinyMaskedLM(vocab_size=tokenizer.vocab_size + 1, d_model=64, n_layers=2, n_heads=4)

# Optimizers
gpt_optim = torch.optim.Adam(gpt.parameters(), lr=0.01)
mlm_optim = torch.optim.Adam(masked_lm.parameters(), lr=0.01)

# Training loop
gpt_losses = []
mlm_losses = []

num_steps = 150

print(f"Training for {num_steps} steps...\n")
print(f"{'Step':<6} {'GPT Loss':<12} {'MLM Loss':<12}")
print("-" * 30)

for step in range(num_steps):
    # --- Train GPT ---
    gpt_optim.zero_grad()
    gpt_logits = gpt(token_ids)
    gpt_loss = F.cross_entropy(
        gpt_logits[:, :-1].reshape(-1, gpt.vocab_size),
        token_ids[:, 1:].reshape(-1)
    )
    gpt_loss.backward()
    gpt_optim.step()
    gpt_losses.append(gpt_loss.item())
    
    # --- Train Masked LM ---
    mlm_optim.zero_grad()
    
    # Create masked version: mask 50% of tokens
    masked_input = token_ids_shifted.clone()
    mask = torch.rand_like(token_ids_shifted, dtype=torch.float) < 0.5
    masked_input[mask] = 0
    
    mlm_logits = masked_lm(masked_input)
    if mask.any():
        mlm_loss = F.cross_entropy(
            mlm_logits[mask].reshape(-1, masked_lm.vocab_size),
            token_ids_shifted[mask].reshape(-1)
        )
        mlm_loss.backward()
        mlm_optim.step()
    else:
        mlm_loss = torch.tensor(0.0)
    
    mlm_losses.append(mlm_loss.item())
    
    if step % 25 == 0:
        print(f"{step:<6} {gpt_loss.item():<12.4f} {mlm_loss.item():<12.4f}")

print("-" * 30)
print(f"{num_steps:<6} {gpt_losses[-1]:<12.4f} {mlm_losses[-1]:<12.4f}")
print(f"\n✓ Both models trained")

### Training Curves

Compare how both models learn over time. Different curves reflect different objectives.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
ax1.plot(gpt_losses, label='GPT (next-token)', linewidth=2, alpha=0.7)
ax1.plot(mlm_losses, label='Masked LM (denoising)', linewidth=2, alpha=0.7)
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Objective Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Zoomed in final portion
ax2.plot(gpt_losses[-50:], label='GPT', linewidth=2, alpha=0.7)
ax2.plot(mlm_losses[-50:], label='Masked LM', linewidth=2, alpha=0.7)
ax2.set_xlabel('Training Step (final 50)')
ax2.set_ylabel('Loss')
ax2.set_title('Final Training Phase')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"GPT final loss: {gpt_losses[-1]:.4f}")
print(f"Masked LM final loss: {mlm_losses[-1]:.4f}")

## Part 4: Generation Comparison

Compare how the two paradigms generate text from the same starting point.

In [ ]:
print("=" * 70)
print("GENERATION COMPARISON")
print("=" * 70)

# Starting prompt
prompt = "hello"
prompt_tokens = torch.tensor([tokenizer.encode(prompt)])
print(f"\nStarting prompt: '{prompt}'")
print(f"Tokens: {prompt_tokens[0].tolist()}")

print("\n" + "=" * 70)
print("GPT: AUTOREGRESSIVE (LEFT-TO-RIGHT)")
print("=" * 70)
print("\nStrategy: Predict next token, append, repeat")
print("- Each token conditions on all previous tokens")
print("- No future context available")
print("- Tokens generated sequentially, left-to-right\n")

for temp in [0, 0.5, 1.0]:
    with torch.no_grad():
        generated = gpt.generate(prompt_tokens, max_new_tokens=15, temperature=temp)
    text = tokenizer.decode(generated[0].tolist())
    print(f"Temperature {temp}: '{text}'")

print("\n" + "=" * 70)
print("MASKED LM: ITERATIVE UNMASKING (DIFFUSION-STYLE)")
print("=" * 70)
print("\nStrategy: Start all masked, progressively unmask confident predictions")
print("- All positions have full context (bidirectional)")
print("- Iteratively refines sequence over multiple steps")
print("- Can 'revise' earlier decisions as context builds\n")

for steps in [2, 4, 8]:
    with torch.no_grad():
        generated = masked_lm.generate(length=20, steps=steps, temperature=1.0)
    # Shift back from vocab_id to token_id
    tokens = [t.item() - 1 for t in generated[0] if t > 0]
    text = tokenizer.decode(tokens)
    print(f"{steps} steps: '{text}'")

## Part 5: Architecture Summary

Final side-by-side comparison of both architectures.

In [ ]:
print("\n" + "=" * 80)
print(" " * 25 + "ARCHITECTURE COMPARISON")
print("=" * 80)

comparison = """
SHARED COMPONENTS (Identical in both models):
  ✓ TokenPositionalEmbedding  — Token + sinusoidal position embeddings
  ✓ MultiHeadSelfAttention    — Scaled dot-product attention
  ✓ FeedForward Networks      — 2-layer Dense + ReLU
  ✓ Layer Normalization       — Pre-norm architecture
  ✓ Residual Connections      — x + attn + ff pattern

DIFFERENCES:
  Training Objective:
    GPT:       Next-token prediction (causal LM)
    Masked LM: Masked token prediction (denoising)
  
  Generation Strategy:
    GPT:       Autoregressive (left-to-right)
                 Predict next token, append, repeat
    Masked LM: Iterative unmasking (diffusion-style)
                 Start masked, progressively unmask confident predictions
  
  Attention Masking:
    GPT:       Causal mask (triangular) — no future context
    Masked LM: No mask — full bidirectional context

THE KEY INSIGHT:
  The SAME transformer machinery works for both generation paradigms.
  Configuration (causal flag) + training signal determine the approach,
  not fundamentally different architectures.
"""

print(comparison)

print("\n" + "=" * 80)
print(" " * 20 + "THESIS: Shared Transformer Foundation")
print("=" * 80)
print("""
This project demonstrates that autoregressive (GPT) and diffusion (masked-LM)
generation are not fundamentally different approaches requiring separate architectures.

Instead, they are different CONFIGURATIONS of the same core transformer machinery:
  - Same attention blocks (just with/without causal masking)
  - Same embeddings
  - Same feedforward networks
  - Different training objectives and sampling strategies

This suggests a continuum of generation approaches, all built on the same foundation.
""")

## Summary

This notebook has demonstrated:

1. **Code Identity** — Both models import and use the exact same `MultiHeadSelfAttention` class and `TokenPositionalEmbedding`.

2. **Single Configuration Flag** — The `causal` parameter is the only architectural difference.

3. **Different Learning Dynamics** — Different training objectives lead to different loss curves, but both converge.

4. **Different Generation Strategies** — Despite sharing the same foundation, the models generate very differently:
   - GPT: Sequential left-to-right prediction
   - Masked LM: Iterative refinement with full context

5. **Unified Framework** — The transformer is a versatile foundation that supports multiple paradigms through configuration changes.

### Implications

- **Modular Design** — Shared modules reduce code duplication and make the codebase maintainable
- **Generalization** — Other generation approaches (prefix LM, encoder-decoder) could reuse the same attention block
- **Research Value** — Easy to swap attention patterns and study their effects on learning and generation
- **Practical Benefit** — Smaller codebase, easier to debug, clearer separation of concerns

### Further Exploration

Potential extensions:
- Add attention visualization during generation
- Compare quality metrics (diversity, coherence) between approaches
- Train on longer sequences and measure generalization
- Implement hybrid attention patterns (e.g., prefix LM: left-to-right for prompt, bidirectional for generation)
- Explore the effect of masking patterns on learning